# Main Fig 4 — Iso-compute: vs-Total + Pareto

**Layout**: 2 rows × N_TASKS cols  
Row 1 = AUROC vs total compute (L×K); Row 2 = Pareto-optimal frontier.

In [ ]:
import sys
from pathlib import Path

# ── Workspace root (parent of NSRR-tools/) ────────────────────────────────────
WORKSPACE_ROOT = Path("../../../../..").resolve()   # adjust if notebook depth differs
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path
sys.path.insert(0, str(Path(".").resolve()))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib
matplotlib.use("Agg")   # comment out in Jupyter to get inline plots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()
print("Setup OK — workspace root:", WORKSPACE_ROOT)

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD   = "transformer"
TASKS  = MAIN_TASKS
METRIC = "auroc"
N_TASKS = len(TASKS)

hmaps = {t: load_heatmap("phase0_v3", t, HEAD) for t in TASKS}

In [ ]:
fig, axes = plt.subplots(2, N_TASKS, figsize=(FULL_W, 4.0))

panel_idx = 0
for col, task in enumerate(TASKS):
    ax_top = axes[0, col]
    ax_bot = axes[1, col]

    panels.vs_total_panel(ax_top, hmaps[task], col=METRIC)
    ax_top.set_title(TASK_LABEL[task], fontsize=8)
    add_panel_label(ax_top, f"({chr(97 + col)})")

    panels.pareto_panel(ax_bot, hmaps[task], col=METRIC)
    add_panel_label(ax_bot, f"({chr(97 + N_TASKS + col)})")

    # Only leftmost column gets y-labels
    if col > 0:
        ax_top.set_ylabel("")
        ax_bot.set_ylabel("")
    if col < N_TASKS - 1:
        ax_top.get_legend().remove() if ax_top.get_legend() else None
        ax_bot.get_legend().remove() if ax_bot.get_legend() else None

fig.tight_layout(h_pad=1.5, w_pad=0.8)
save_figure(fig, FINAL_OUT, "main_fig4_iso_main")
plt.show()